# alignment.ipynb

TCFN: MLP projector alignment (negative result, see report Sec. 3.4-3.6, 6.2).

Paths are relative: `data/` holds dialogues and embeddings (not included), `models/` holds fine-tuned weights (not included). See the README.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

## Alignment network (MLP projector)

In [ ]:
class AlignmentNet(nn.Module):
    def __init__(self, d_local, d_victim, hidden_dim=512):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(d_local, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),

            nn.Linear(hidden_dim, d_victim),
        )

        self.skip = nn.Linear(d_local, d_victim)

    def forward(self, x):
        """
        x: [B, d_local]
        return: [B, d_victim]
        """

        out = self.mlp(x) + self.skip(x)

        out = F.normalize(out, dim=-1)

        return out

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, local_embs, victim_embs):
        assert len(local_embs) == len(victim_embs)
        self.local = local_embs
        self.victim = victim_embs

    def __len__(self):
        return len(self.local)

    def __getitem__(self, idx):
        return self.local[idx], self.victim[idx]

In [ ]:
def get_dataloader(local_embs, victim_embs, batch_size=64, shuffle=False):
    dataset = EmbeddingDataset(local_embs, victim_embs)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=True
    )

    return loader

In [ ]:
def alignment_loss(pred, target, alpha=0.5):
    """
    pred:   [B, d]
    target: [B, d]
    """

    pred_n = F.normalize(pred, dim=-1)
    target_n = F.normalize(target, dim=-1)
    cosine_loss = 1 - (pred_n * target_n).sum(dim=-1).mean()

    mse_loss = F.mse_loss(pred, target)

    return alpha * mse_loss + (1 - alpha) * cosine_loss

def train(model, dataloader, device, lr=1e-4, epochs=70):
    model.to(device)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0

        for local_batch, victim_batch in dataloader:
            local_batch = local_batch.to(device)
            victim_batch = victim_batch.to(device)

            pred = model(local_batch)

            loss = alignment_loss(pred, victim_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.6f}")

## Training data and training

In [ ]:
def cat_data(local_data, victim_data):
  local = []
  victim = []
  for i, item1 in enumerate(local_data):
    item2 = victim_data[i]
    for j, d1 in enumerate(item1['dialogue']):
      local.append(torch.tensor(d1['embedding']))
      d2 = item2['dialogue'][j]
      victim.append(torch.tensor(d2['embedding']))
  return local, victim

In [ ]:
alignModel = AlignmentNet(d_local=768, d_victim=768, hidden_dim=512).to(device)

In [ ]:
import json
with open("data/ill_embed.json", "r") as f:
  ill_v = json.load(f)
with open("data/bank_embed.json", "r") as f:
  bank_v = json.load(f)
with open("data/new_user_profile_embed.json", "r") as f:
  user_v = json.load(f)

with open("data/ill_localembed.json", "r") as f:
  ill_l = json.load(f)
with open("data/bank_localembed.json", "r") as f:
  bank_l = json.load(f)
with open("data/new_user_profile_localembed.json", "r") as f:
  user_l = json.load(f)

il, iv = cat_data(ill_l[:79], ill_v[:79])
bl, bv = cat_data(bank_l[:223], bank_v[:223])
ul, uv = cat_data(user_l[:61], user_v[:61])
local = ul + bl + il
victim = uv + bv + iv
local = torch.stack(local).to(device)
victim = torch.stack(victim).to(device)
print("Dataset ready\n")
loader = get_dataloader(local, victim)
train(alignModel, loader, device)

## Models for recovery

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

llm_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(llm_name)

local_embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)

victim_embedder = SentenceTransformer("intfloat/e5-base-v2", device=device)

llm.to(device)
llm.eval()

## Recovery with the Transformer context encoder

In [ ]:
for p in local_embedder.parameters():
    p.requires_grad_(False)

In [ ]:
def encode_local(texts):
    return local_embedder.encode(
        texts,
        convert_to_tensor=True,
        device=device,
        show_progress_bar=False,
    )

def format_context(prev_turns):
    """
    Concatenate the previously recovered turns in dialogue format
    Adjust to the chat template of the LLM in use
    """
    formatted = ""
    for i, turn in enumerate(prev_turns):
        if i % 2 == 0:
            formatted += f"{turn}"
        else:
            formatted += f"{turn}"
    return formatted

def diversity_filter(token_embs, KS, T_hw):
    top_k = token_embs.size(0)
    selected_indices = []
    mask = torch.ones(top_k, dtype=torch.bool, device=token_embs.device)

    for i in range(top_k):
        if not mask[i]:
            continue
        selected_indices.append(i)
        if len(selected_indices) >= KS:
            break
        sims = F.cosine_similarity(token_embs[i].unsqueeze(0), token_embs, dim=1)
        sims[i] = 0
        mask = mask & (sims <= T_hw)
    return selected_indices

In [ ]:
def build_context_emb_transformer(prev_turns, window_llm=5):
    """
    prev_turns: list[str], previously recovered turns
    window_llm: only the most recent turns go into the LLM prefix, to bound the input length

    Returns:
        context_emb: [1, d_local], history embedding encoded by the Transformer
        llm_prefix: str, prefix text given to the LLM
    """
    if not prev_turns:
        return None, " "

    with torch.no_grad():
        turn_embs = encode_local(prev_turns)
        context_emb = alignModel(turn_embs).unsqueeze(0)

    recent_turns = prev_turns[-window_llm:]
    llm_prefix = format_context(recent_turns)

    return context_emb, llm_prefix

In [ ]:
import string
allowed_chars = set(string.ascii_letters + string.digits + " .,!?'\"-")

In [ ]:
def zscore(x):
    return (x - x.mean()) / (x.std() + 1e-8)

In [ ]:
def new_train_step_multiturn(target_embs, max_len, prev_turns,
                         alpha=1.0, beta=0.05, gamma=0.3):

    batch_size = len(target_embs)
    topk = 2000
    KS = 1000
    T_hw = 0.9
    beam_width = 5

    context_emb, llm_prefix = build_context_emb_transformer(prev_turns)

    beams = [[(llm_prefix, 0.0)] for _ in range(batch_size)]

    for t in range(max_len):
        batch_prefix_idx = []
        batch_prefix = []
        batch_prefix_score = []

        for b_idx, beam_list in enumerate(beams):
            for prefix, score in beam_list:
                batch_prefix_idx.append(b_idx)
                batch_prefix.append(prefix)
                batch_prefix_score.append(score)

        if not batch_prefix:
            break

        batch_input_ids = tokenizer(
            batch_prefix, return_tensors="pt", padding=True
        ).input_ids.to(device)

        with torch.no_grad():
            batch_logits = llm(batch_input_ids).logits[:, -1, :]
        batch_log_probs = F.log_softmax(batch_logits, dim=-1)

        all_new_texts = []
        all_candidate_meta = []

        for i, b_idx in enumerate(batch_prefix_idx):

            prefix = batch_prefix[i]
            prev_score = batch_prefix_score[i]

            top_ids = batch_log_probs[i].topk(topk, dim=-1).indices

            with torch.no_grad():
                token_embs = llm.get_input_embeddings()(top_ids)

            diverse_indices = diversity_filter(token_embs, KS=KS, T_hw=T_hw)

            for idx in diverse_indices:

                token_id = top_ids[idx].item()
                lang_score = batch_log_probs[i][token_id]

                token_str = tokenizer.decode(token_id, skip_special_tokens=True)
                if not all(c in allowed_chars for c in token_str):
                  continue

                new_text = prefix + (" " if not token_str.startswith(" ") else "") + token_str

                all_new_texts.append(new_text)

                all_candidate_meta.append((b_idx, new_text, lang_score, prev_score))

        if not all_new_texts:
            break

        with torch.no_grad():
            all_local_embs = encode_local(all_new_texts)
            all_pred_v = alignModel(all_local_embs)

        candidate_scores = []

        for k, (b_idx, new_text, lang_score, prev_score) in enumerate(all_candidate_meta):

            pred_v = all_pred_v[k]
            target_v = target_embs[b_idx]

            pred_v = F.normalize(pred_v, dim=-1)
            target_v = F.normalize(target_v, dim=-1)

            pred_score = F.cosine_similarity(pred_v, target_v, dim=-1)

            if context_emb is not None:
                sim = F.cosine_similarity(pred_v.unsqueeze(0), context_emb, dim=-1)
                context_score = sim.mean()
            else:
                context_score = 0.0

            pred_score = zscore(pred_score)
            lang_score = zscore(lang_score)

            final_score = (
                prev_score +
                alpha * pred_score +
                beta  * lang_score +
                gamma * context_score
            )

            candidate_scores.append((final_score, b_idx, new_text))

        from collections import defaultdict
        grouped = defaultdict(list)

        for score, b_idx, new_text in candidate_scores:
            grouped[b_idx].append((score, new_text))

        new_beams = [[] for _ in range(batch_size)]

        for b_idx in range(batch_size):

            top_candidates = sorted(
                grouped[b_idx],
                key=lambda x: x[0],
                reverse=True
            )[:beam_width]

            new_beams[b_idx] = [
                (text, score) for score, text in top_candidates
            ]

        beams = new_beams

        print([[t for t, _ in b] for b in beams])

    results = []

    for b_idx in range(batch_size):
        best = beams[b_idx][0][0] if beams[b_idx] else ""
        recovered = best[len(llm_prefix):].strip()
        results.append(recovered)

    return results

In [ ]:
def recover_multiturn(all_leaked_embs, max_len):
    recovered_turns = []

    for turn_idx, leaked_emb in enumerate(all_leaked_embs):
        print(f"Recovering turn {turn_idx + 1} / {len(all_leaked_embs)}...")

        result = new_train_step_multiturn(
            target_embs=[leaked_emb.unsqueeze(0)],
            max_len=max_len,
            prev_turns=recovered_turns,
        )

        recovered_turns.append(result[0])
        print(f"Recovered turn {turn_idx + 1}: {result[0]}\n")

    return recovered_turns

## Example

In [ ]:
# Qualitative example: reconstruct one medical dialogue
for item in ill_v[38:39]:
  embed = []
  max_l = []
  for d in item['dialogue']:
    text = d['text']
    length = int(len(text.split()) * 1.2)
    max_l.append(length)
    embed.append(torch.tensor(d['embedding']).to(device))
  recover_multiturn(embed, max(max_l))